# Activity: Postman Websocket Echo Server 
Connect to the Postman WebSocket echo service and watch the server send back whatever you transmit.

> __Learning Objectives__ 
> 
> By the end of this activity, you will be able to:
> - Configure a Julia WebSocket client for a basic echo service.
> - Send structured payloads over a WebSocket connection and receive responses.
> - Control message loops and cleanly close WebSocket connections after collecting data.

Let's get started!
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Constants
We define the Postman WebSocket URL, the message to echo, and a small limit on how many responses to capture before closing the connection.


In [5]:
url = "wss://ws.postman-echo.com/raw"; # WebSocket URL for Postman Echo
message_text = "Hello from Julia!"; # Text to send to the echo server
number_of_messages = 2; # Number of echoed messages to receive before closing
const WS = HTTP.WebSockets; # Create alias for WebSockets module for convenience


___
## Task 1: Prepare the echo payload
The Postman echo server simply returns whatever we send. We will craft a small JSON payload so we can test sending structured data as a single message.


In [6]:
payload = JSON.json(Dict(
    "event" => "echo",
    "message" => message_text,
)); # JSON payload to send

payload


"{\"event\":\"echo\",\"message\":\"Hello from Julia!\"}"

___
## Task 2: Implement the WebSocket echo loop
Open a WebSocket connection to Postman, send the payload once on connect, and log the messages that come back. We normalize incoming data to a String, attempt to parse JSON echoes, and close the socket after the desired count.


In [ ]:
WS.open(url) do ws
    WS.send(ws, payload) # send payload once when the socket opens
    seen = 0; # count of messages seen
    max_messages = number_of_messages; # number of messages to receive (then we close the connection)

    for raw in ws
        s = raw isa AbstractString ? raw : String(raw) # raw can be String or Vector{UInt8}; normalize to String

        parsed = try
            JSON.parse(s)
        catch err
            nothing # not valid JSON; treat as plain text
        end

        if parsed !== nothing
            @info "json echo" parsed
        else
            @info "text echo" message=s
        end

        seen += 1 # update message count
        if seen >= max_messages
            @info "max message count reached; closing websocket" count=seen
            WS.close(ws) # close the WebSocket connection
            break
        end
    end
end


┌ Info: json echo
│   parsed = JSON.Object{String, Any}("event" => "echo", "message" => "Hello from Julia!")
└ @ Main /Users/jdv27/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-142/module-3/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X21sZmlsZQ==.jl:16


___
## Summary
In this activity we connected to the Postman WebSocket echo service, sent a structured JSON payload, and received the echoed response in Julia.

> __Key Takeaways:__
> 
> - Use `HTTP.WebSockets.open` to establish and manage WebSocket sessions.
> - Normalize and parse incoming frames so you can handle both strings and byte arrays.
> - Control your receive loop with counters to close the socket deterministically.

WebSockets make it straightforward to test connectivity by echoing data before moving to richer real-time APIs.
___
